In [ ]:
from azure.ai.ml import MLClient
from azure.ai.ml.entities import Data
from azure.ai.ml.constants import AssetTypes
from azure.identity import DefaultAzureCredential
import pandas as pd
import numpy as np
import requests
import gzip
import shutil
import time
import myvariant
import mygene

In [ ]:
url = "https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh38/clinvar.vcf.gz"
vcf_gz = "clinvar.vcf.gz"
vcf_file = "clinvar.vcf"

print("[+] Downloading ClinVar VCF...")
with requests.get(url, stream=True) as r:
    r.raise_for_status()
    with open(vcf_gz, "wb") as f:
        for chunk in r.iter_content(chunk_size=8192):
            f.write(chunk)
print("[✓] File Downloaded:", vcf_gz)

print("[+] Unboxing file...")
with gzip.open(vcf_gz, "rb") as f_in:
    with open(vcf_file, "wb") as f_out:
        shutil.copyfileobj(f_in, f_out)
print("[✓] Unboxed too:", vcf_file)


In [ ]:
def parse_vcf(vcf_path, limit=None):
    """
    Parser tekstowy pliku VCF (np. ClinVar) do pandas DataFrame.
    Rozbija INFO na osobne kolumny, uzupełnia brakujące BASE_COLUMNS.
    """
    def parse_info(info_str):
        parsed = {}
        for item in info_str.split(';'):
            if '=' in item:
                key, value = item.split('=')
                parsed[key] = value
            else:
                parsed[item] = True
        return parsed

    records = []
    headers = []

    with open(vcf_path, 'r', encoding='UTF-8', errors='replacce') as file:
        for line in file:
            if line.startswith('##'):
                continue
            elif line.startswith('#CHROM'):
                headers = line.strip().lstrip('#').split('\t')
                continue

            row = line.strip().split('\t')
            record = dict(zip(headers, row))

            if 'INFO' in record:
                record.update(parse_info(record['INFO']))

            records.append(record)
            if limit and len(records) >= limit:
                break

    return pd.DataFrame(records)

def standarize_df(df):
    keep_list = [
    'CHROM', 'POS', 'REF', 'ALT', 
        
        'MC',         
        'GENEINFO',   
        'CLNVC',      
        'AF_EXAC', 'AF_TGP', 'AF_ESP',
        'ORIGIN',     
        'CLNREVSTAT', 
        'CLNDN']
    labels = df['CLNSIG'] if 'CLNSIG' in df.columns else pd.Series([None] * len(df))
    features_df =  df.drop(columns=['CLNSIG'], errors ='ignore')
    existing_keep_cols = [col for col in keep_list if col in features_df.columns]
    missing_cols = [col for col in keep_list if col not in features_df.columns]
    if missing_cols:
      print(f'[WARNING] Can not find: {missing_cols}')
    df_filtered = features_df[existing_keep_cols]

    return df_filtered, labels


def process_input(input_type, **kwargs):
    if input_type == 'vcf':
        vcf_path = kwargs.get('vcf_path')
        limit = kwargs.get('limit', None)
        print(f'[INFO] Reading file: {vcf_path} (limit={limit})')
        df = parse_vcf(vcf_path, limit)
        return standarize_df(df)
    else:
        raise ValueError('Not known input type!')

In [ ]:
df_feautures, df_labels = process_input(
    input_type="vcf",
    vcf_path = 'clinvar.vcf',
    limit=None
)
df_full = df_feautures.copy()
df_full['CLNSIG'] = df_labels.copy()

In [ ]:
df_full.info()

In [ ]:
mapping_acmg = {
    #BENIGN
    "Benign": 0,
    "Benign/Likely_benign": 0,
    "Benign|risk_factor": 0,
    "Benign|other": 0,
    "Benign|association": 0,
    "Benign/Likely_benign|other": 0,
    "Benign|Affects": 0,
    "Benign|drug_response": 0,
    "Benign|protective": 0,

    #LIKELY BENIGN
    "Likely_benign": 0,
    "Likely_benign|other": 0,
    "Likely_benign|drug_response": 0,
    "Likely_benign|drug_response|other": 0,

    #UNCERTAIN SIGNIFICANCE (VUS)
    "Uncertain_significance": -1,
    "Conflicting_classifications_of_pathogenicity": -1,
    "not_provided": -1,
    "no_classification_for_the_single_variant": -1,
    "no_classifications_from_unflagged_records": -1,
    "other": -1,
    "association": -1,
    "association_not_found": -1,
    "Uncertain_significance/Uncertain_risk_allele": -1,
    "Uncertain_risk_allele": -1,
    "Uncertain_significance|other": -1,
    "Uncertain_significance|risk_factor": -1,
    "Uncertain_significance|drug_response": -1,
    "Uncertain_significance|association": -1,
    "Conflicting_classifications_of_pathogenicity|other": -1,
    "Conflicting_classifications_of_pathogenicity|risk_factor": -1,
    "Conflicting_classifications_of_pathogenicity|drug_response": -1,
    "Conflicting_classifications_of_pathogenicity|association": -1,

    # 4️ LIKELY PATHOGENIC
    "Likely_pathogenic": 1,
    "Likely_pathogenic|drug_response": 1,
    "Likely_pathogenic|risk_factor": 1,
    "Likely_pathogenic/Likely_risk_allele": 1,
    "Likely_pathogenic|association": 1,
    "Likely_pathogenic,_low_penetrance": 1,

    # PATHOGENIC
    "Pathogenic": 1,
    "Pathogenic/Likely_pathogenic": 1,
    "Pathogenic|other": 1,
    "Pathogenic|drug_response": 1,
    "Pathogenic|risk_factor": 1,
    "Pathogenic/Likely_risk_allele": 1,
    "Pathogenic|Affects": 1,
    "Pathogenic/Likely_pathogenic|risk_factor": 1,
    "Pathogenic/Likely_pathogenic|other": 1,
    "Pathogenic/Likely_pathogenic/Pathogenic,_low_penetrance": 1,
    "Pathogenic/Likely_pathogenic/Likely_risk_allele": 1
}

In [ ]:
df_full['CLNSIG'] = df_full['CLNSIG'].map(mapping_acmg)

In [ ]:
df_full['CLNSIG'].isna().mean()

In [ ]:
df_full = df_full.dropna(subset=['CLNSIG'])

In [ ]:
df_full['CLNSIG'].isna().mean()

In [ ]:
df_full['CLNSIG'].value_counts()

In [ ]:
df_vus = df_full[df_full['CLNSIG'] == -1.0].copy()
df = df_full[df_full['CLNSIG'] != -1.0].copy()

In [ ]:
df_vus.head() 

In [ ]:
df['CLNSIG'].value_counts()

In [ ]:
samples_per_class = 10_000

df_balanced = df.groupby('CLNSIG').apply(
        lambda x: x.sample(min(len(x), samples_per_class), random_state=42), 
        include_groups=False).reset_index(level=0)    

In [ ]:
df_balanced['CLNSIG'].value_counts()

In [ ]:
df_balanced['CHROM'].value_counts()


In [ ]:
def annotate_with_myvariant(df):
    mv = myvariant.MyVariantInfo()
    
    print("[+] Generating HGVS ID for GRCh38...")
    
    
    def make_hgvs(row):
        chrom = str(row['CHROM']).strip()
        if chrom == 'MT': chrom = 'M'
        if not chrom.startswith('chr'): chrom = f"chr{chrom}"
        return f"{chrom}:g.{str(row['POS']).strip()}{str(row['REF']).strip()}>{str(row['ALT']).strip()}"

    hg38_ids = df.apply(make_hgvs, axis=1).tolist()
    fields = [
        'gnomad_exome.af.af',
        'cadd.phred',                     
        'dbnsfp.phylop.100way_vertebrate.score', 
        'dbnsfp.revel.score',             
        'dbnsfp.interpro.domain',
        'gnomad_exome.constraints.pli',
        'dbnsfp.spliceai_ds_max',         
        'dbnsfp.dbscsnv.ada_score',       
        'dbnsfp.dbscsnv.rf_score'
    ]
    
    batch_size = 1000
    all_results = []
    total = len(hg38_ids)
    
    
    for i in range(0, total, batch_size):
        batch = hg38_ids[i:i+batch_size]
        try:
            res = mv.getvariants(batch, fields=",".join(fields), assembly='hg38', as_dataframe=True)
            
            if not res.empty:
                if 'notfound' in res.columns:
                    res = res[res['notfound'].isna()]
                res.index.name = 'query_id'
                res = res.reset_index()
                all_results.append(res)
                
                print(f"   -> Partia {i}-{min(i+batch_size, total)} OK")
            
        except Exception as e:
            print(f"   [!] Error in  {i}: {e}")
        
        time.sleep(0.2)

    if not all_results:
        df_final = df.copy()
        for col in fields:
            df_final[col] = np.nan
        return df_final

    df_anno = pd.concat(all_results)
    df_anno = df_anno.drop_duplicates(subset=['query_id'])
    
    for col in fields:
        if col not in df_anno.columns:
            df_anno[col] = np.nan


    def clean_value(x):
        if isinstance(x, list):
            valid_x = [v for v in x if pd.notna(v)]
            if not valid_x: return np.nan
            try:
                return max(valid_x) 
            except:
                return valid_x[0]
        return x

    cols_to_clean = [
        'cadd.phred', 
        'dbnsfp.revel.score', 
        'dbnsfp.spliceai_ds_max', 
        'gnomad_exome.constraints.pli', 
        'dbnsfp.phylop.100way_vertebrate.score',
        'gnomad_exome.af.af'
    ]
    
    for col in cols_to_clean:
        if col in df_anno.columns:
            df_anno[col] = df_anno[col].apply(clean_value)
            df_anno[col] = pd.to_numeric(df_anno[col], errors='coerce')

    df['hg38_temp'] = hg38_ids
    df_final = df.merge(df_anno, left_on='hg38_temp', right_on='query_id', how='left')
    
    cols_drop = ['hg38_temp', 'query_id', '_id', '_version', 'query', 'notfound']
    df_final.drop(columns=[c for c in cols_drop if c in df_final.columns], inplace=True, errors='ignore')
    
    return df_final

In [ ]:
df_full_annotate = annotate_with_myvariant(df_balanced)


In [ ]:
df_full_annotate.head()


In [ ]:
df_full_annotate.info()


In [ ]:
df_full_annotate = df_full_annotate.drop(columns=['dbnsfp.dbscsnv.rf_score', 'dbnsfp.dbscsnv.ada_score', 'dbnsfp.spliceai_ds_max', 'gnomad_exome.constraints.pli', 'cadd.phred', 'gnomad_exome._license', 'dbnsfp._license'], axis=1)
df_full_annotate.info()

In [ ]:
df_full_annotate['GENEINFO'].value_counts()


In [ ]:
df_final = df_full_annotate.copy()


In [ ]:
def extract_gene_symbol(s):
    if pd.isna(s): return None
    return str(s).split(':')[0] 

df_final['GENE_SYMBOL'] = df_final['GENEINFO'].apply(extract_gene_symbol)
unique_genes = df_final['GENE_SYMBOL'].dropna().unique().tolist()

mg = mygene.MyGeneInfo()
gene_res = mg.querymany(unique_genes, scopes='symbol', fields='gnomad.pli,gnomad.loeuf', as_dataframe=True)
gene_res = gene_res.reset_index()
target_cols = ['gnomad.pli', 'gnomad.loeuf']

for col in target_cols:
    if col not in gene_res.columns:
        gene_res[col] = np.nan

final_cols = ['query'] + target_cols
gene_res = gene_res[final_cols]

gene_res.rename(columns={
    'query': 'GENE_SYMBOL', 
    'gnomad.pli': 'GENE_pLI', 
    'gnomad.loeuf': 'GENE_LOEUF'
}, inplace=True)

gene_res = gene_res.groupby('GENE_SYMBOL').mean().reset_index()
df_enriched = df_final.merge(gene_res, on='GENE_SYMBOL', how='left')

In [ ]:
def prepare_domain_for_parquet(val):
    if isinstance(val, (list, np.ndarray)):
        if len(val) == 0: 
            return np.nan
        return "|".join(map(str, val))
    if pd.isna(val):
        return np.nan
    return str(val)

if 'dbnsfp.interpro.domain' in df_enriched.columns:
    df_enriched['dbnsfp.interpro.domain'] = df_enriched['dbnsfp.interpro.domain'].apply(prepare_domain_for_parquet)

df_enriched.to_parquet('clinvar_final_enriched.parquet', index=False)


In [ ]:
ml_client = MLClient.from_config(credential=DefaultAzureCredential())
data_name = 'clinvar_enriched'
data_version = '1.0.0'
my_data = Data(
    path='./clinvar_final_enriched.parquet',
    type=AssetTypes.URI_FILE,
    description='Wzbogacone dane ClinVar',
    name=data_name,
    version=data_version
)

try:
    ml_client.data.create_or_update(my_data)
except Exception as e:
    print(f'Error: {e}')